In [1]:
# Getting Data from Github
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-adverse-events.csv


# Verify the files are downloaded
!ls -lh 2-adverse-events.csv

-rw-r--r-- 1 root root 588K Feb  9 22:55 2-adverse-events.csv


In [2]:
# Import the needed tools to configure and control the system from pyspark library
from pyspark import SparkConf, SparkContext

# Create a 'Configuration' object to define how the program should run
# .setMaster("local") is commanding the code to run on your own computer
# .setAppName("Adverse_Event") gives the specific task a name for the logs
conf = SparkConf().setMaster("local").setAppName("Adverse_Arm")

# Creating the 'Spark Context', which is an entry point for all Spark functions
# It takes the configuration, and starts the engine
sc = SparkContext(conf = conf)

In [3]:
# Define a function to parse each line of the CSV file
def parseLine(filterLine):
    # Split the line by commas
    fields = filterLine.split(',')
    # Extract the adverse_event and arm_affected
    adverseEvent = str(fields[1])
    # Convert arm_affected to int, handling potential errors for non-numeric values gracefully (e.g., header)
    armAffected= int(fields[2])
    # Return a tuple (adverse_event, arm_affected)
    return (adverseEvent, armAffected)

# Load the CSV file as an RDD
lines = sc.textFile("2-adverse-events.csv")

filterLine = lines.filter(lambda x: "allergic_ID" not in x)
rdd = filterLine.map(parseLine)
eventMap = rdd.mapValues(lambda x: (x, 1))
Adverse_Arm = eventMap.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
results = Adverse_Arm.collect()

#Sort the result in the numerical order of arm_affected
results.sort(key=lambda x: x[1])

# Print the results
# For each result, print the adverse event (string) and arm affected (integer)
for result in results:
    # Assuming result[0] is adverse_event (string) and result[1] is arm_affected (int)
    print(result)

# Stop the SparkContext
sc.stop()

('drug_hypersensitivity', (1, 1))
('hypersensitivity', (1, 1))
('allergy', (4, 3))
('DERMATITIS_ALLERGIC', (4, 4))
('Dermatitis_Allergic', (11, 10))
('DRUG_HYPERSENSITIVITY', (45, 27))
('Allergy', (72, 19))
('ALLERGIC_REACTION', (81, 13))
('allergic_reaction', (106, 28))
('Allergic_Reaction', (145, 76))
('Dermatitis_allergic', (211, 151))
('HYPERSENSITIVITY', (241, 82))
('Drug_Hypersensitivity', (818, 145))
('Drug_hypersensitivity', (1029, 326))
('Anaphylaxis', (1088, 111))
('Allergic_reaction', (2291, 335))
('Hypersensitivity', (2493, 755))
